In [1]:
from pathlib import Path
project_root = Path.cwd().parent
project_root

WindowsPath('e:/capstone_project/mortgage-model-reliability')

In [2]:
freddie_raw_dir = project_root / "data" / "raw" / "Freddie_Mac"

freddie_raw_dir

WindowsPath('e:/capstone_project/mortgage-model-reliability/data/raw/Freddie_Mac')

In [3]:
freddie_raw_dir.exists()

True

In [4]:
list(freddie_raw_dir.iterdir())

[WindowsPath('e:/capstone_project/mortgage-model-reliability/data/raw/Freddie_Mac/file_headers_july_2026.zip'),
 WindowsPath('e:/capstone_project/mortgage-model-reliability/data/raw/Freddie_Mac/file_layout_july_2026.xlsx'),
 WindowsPath('e:/capstone_project/mortgage-model-reliability/data/raw/Freddie_Mac/release-47-sample-files'),
 WindowsPath('e:/capstone_project/mortgage-model-reliability/data/raw/Freddie_Mac/release-47-sample-files.zip'),
 WindowsPath('e:/capstone_project/mortgage-model-reliability/data/raw/Freddie_Mac/sample_2015'),
 WindowsPath('e:/capstone_project/mortgage-model-reliability/data/raw/Freddie_Mac/sample_2015.zip')]

# Navigate to the actual sample files

In [5]:
sample_dir = freddie_raw_dir / "release-47-sample-files" / "Sample Files"

sample_dir.exists()

True

In [6]:
[item.name for item in sample_dir.iterdir()]

['origination_sample_file.txt', 'performance_sample_file.txt', 'scp46436']

In [7]:
origination_file = sample_dir / "origination_sample_file.txt"

origination_file.exists()

True

In [8]:
with open(origination_file, "r") as file:
    for _ in range(3):
        print(file.readline())

740|200007|N|203006||0|1|S|65|21|253000.00|65|7.875|T|N|FRM|OR|CO|977|F26Q50000001|P|360|1|WASHINGTON MUTUAL BANK|N|||N|7|N|9999

777|200006|N|203005|16974|0|1|I|80|44|122000.00|80|9.250|R|N|FRM|IL|CO|606|F26Q50000002|P|360|1|BANK OF AMERICA, N.A.|N|||N|7|N|9999

651|200009|N|203008|37964|0|1|P|72|34|150000.00|72|8.375|R|N|FRM|PA|SF|190|F26Q50000003|C|360|2|OTHER|N|||N|7|N|9999



# after downloading file header and file layout folders from website

In [9]:
[item.name for item in freddie_raw_dir.iterdir()]

['file_headers_july_2026.zip',
 'file_layout_july_2026.xlsx',
 'release-47-sample-files',
 'release-47-sample-files.zip',
 'sample_2015',
 'sample_2015.zip']

In [10]:
from zipfile import ZipFile

In [11]:
headers_zip = freddie_raw_dir / "file_headers_july_2026.zip"

with ZipFile(headers_zip, "r") as zip_file:
    print(zip_file.namelist())

['origination_data_file_header.txt', 'performance_data_file_header.txt']


# Read the origination column names

In [12]:
with ZipFile(headers_zip, "r") as zip_file:
    with zip_file.open("origination_data_file_header.txt") as file:
        origination_header = file.read().decode("utf-8")

print(origination_header)

CLASSIC FICO|FIRST PAYMENT DATE|FIRST TIME HOMEBUYER INDICATOR|MATURITY DATE|METROPOLITAN STATISTICAL AREA (MSA) OR METROPOLITAN DIVISION|MORTGAGE INSURANCE PERCENTAGE (MI %)|NUMBER OF UNITS|OCCUPANCY STATUS|ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)|ORIGINAL DEBT-TO-INCOME (DTI) RATIO|ORIGINAL UPB|ORIGINAL LOAN-TO-VALUE (LTV)|ORIGINAL INTEREST RATE|CHANNEL|PREPAYMENT PENALTY INDICATOR|AMORTIZATION TYPE|PROPERTY STATE|PROPERTY TYPE|POSTAL CODE|LOAN IDENTIFIER|LOAN PURPOSE|ORIGINAL LOAN TERM|NUMBER OF BORROWERS|SELLER NAME|SUPER CONFORMING FLAG|PRE-HARP LOAN SEQUENCE NUMBER|SPECIAL ELIGIBILITY PROGRAM|HARP INDICATOR|PROPERTY VALUATION METHOD|INTEREST ONLY (I/O) INDICATOR|VANTAGESCORE 4.0


# Do the header and data actually match

In [13]:
origination_columns = origination_header.strip().split("|")

len(origination_columns)

31

In [14]:
with open(origination_file, "r") as file:
    first_record = file.readline().strip().split("|")

len(first_record)

31

# Load the origination sample into pandas

In [15]:
import pandas as pd

In [16]:
origination_df = pd.read_csv(
    origination_file,
    sep="|",
    names=origination_columns
)

origination_df.head()

,CLASSIC FICO,FIRST PAYMENT DATE,FIRST TIME HOMEBUYER INDICATOR,MATURITY DATE,METROPOLITAN STATISTICAL AREA (MSA) OR METROPOLITAN DIVISION,MORTGAGE INSURANCE PERCENTAGE (MI %),NUMBER OF UNITS,OCCUPANCY STATUS,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),ORIGINAL DEBT-TO-INCOME (DTI) RATIO,...,ORIGINAL LOAN TERM,NUMBER OF BORROWERS,SELLER NAME,SUPER CONFORMING FLAG,PRE-HARP LOAN SEQUENCE NUMBER,SPECIAL ELIGIBILITY PROGRAM,HARP INDICATOR,PROPERTY VALUATION METHOD,INTEREST ONLY (I/O) INDICATOR,VANTAGESCORE 4.0
0,740,200007,N,203006,NaN,0,1,S,65,21,...,360,1,WASHINGTON MUTUAL BANK,N,NaN,NaN,N,7,N,9999
1,777,200006,N,203005,16974.0,0,1,I,80,44,...,360,1,"BANK OF AMERICA, N.A.",N,NaN,NaN,N,7,N,9999
2,651,200009,N,203008,37964.0,0,1,P,72,34,...,360,2,OTHER,N,NaN,NaN,N,7,N,9999
3,723,200008,Y,203007,12060.0,17,1,P,90,50,...,360,1,"PRINCIPAL RESIDENTIAL MORTGAGE, INC.",N,NaN,NaN,N,7,N,9999
4,664,200010,N,203009,NaN,0,1,S,71,999,...,360,1,"WELLS FARGO HOME MORTGAGE, INC.",N,NaN,NaN,N,7,N,9999


In [17]:
origination_df.shape

(1000, 31)

In [18]:
origination_df.columns.tolist()

['CLASSIC FICO',
 'FIRST PAYMENT DATE',
 'FIRST TIME HOMEBUYER INDICATOR',
 'MATURITY DATE',
 'METROPOLITAN STATISTICAL AREA (MSA) OR METROPOLITAN DIVISION',
 'MORTGAGE INSURANCE PERCENTAGE (MI %)',
 'NUMBER OF UNITS',
 'OCCUPANCY STATUS',
 'ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)',
 'ORIGINAL DEBT-TO-INCOME (DTI) RATIO',
 'ORIGINAL UPB',
 'ORIGINAL LOAN-TO-VALUE (LTV)',
 'ORIGINAL INTEREST RATE',
 'CHANNEL',
 'PREPAYMENT PENALTY INDICATOR',
 'AMORTIZATION TYPE',
 'PROPERTY STATE',
 'PROPERTY TYPE',
 'POSTAL CODE',
 'LOAN IDENTIFIER',
 'LOAN PURPOSE',
 'ORIGINAL LOAN TERM',
 'NUMBER OF BORROWERS',
 'SELLER NAME',
 'SUPER CONFORMING FLAG',
 'PRE-HARP LOAN SEQUENCE NUMBER',
 'SPECIAL ELIGIBILITY PROGRAM',
 'HARP INDICATOR',
 'PROPERTY VALUATION METHOD',
 'INTEREST ONLY (I/O) INDICATOR',
 'VANTAGESCORE 4.0']

In [19]:
origination_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 31 columns):
 #   Column                                                        Non-Null Count  Dtype  
---  ------                                                        --------------  -----  
 0   CLASSIC FICO                                                  1000 non-null   int64  
 1   FIRST PAYMENT DATE                                            1000 non-null   int64  
 2   FIRST TIME HOMEBUYER INDICATOR                                1000 non-null   str    
 3   MATURITY DATE                                                 1000 non-null   int64  
 4   METROPOLITAN STATISTICAL AREA (MSA) OR METROPOLITAN DIVISION  738 non-null    float64
 5   MORTGAGE INSURANCE PERCENTAGE (MI %)                          1000 non-null   int64  
 6   NUMBER OF UNITS                                               1000 non-null   int64  
 7   OCCUPANCY STATUS                                              1000 non-null   str

# Read the performance header

In [20]:
with ZipFile(headers_zip, "r") as zip_file:
    with zip_file.open("performance_data_file_header.txt") as file:
        performance_header = file.read().decode("utf-8")

print(performance_header)

LOAN IDENTIFIER|PERIOD|CURRENT ACTUAL UPB|CURRENT LOAN DELINQUENCY STATUS|LOAN AGE|REMAINING MONTHS TO LEGAL MATURITY|UNDERWRITING DEFECT AND MAJOR SERVICING DEFECT SETTLEMENT DATE|MODIFICATION FLAG|ZERO BALANCE CODE|ZERO BALANCE EFFECTIVE DATE|CURRENT INTEREST RATE|CURRENT NON-INTEREST BEARING UPB|DUE DATE OF LAST PAID INSTALLMENT (DDLPI)|MI RECOVERIES|NET SALES PROCEEDS|NON MI RECOVERIES|TOTAL EXPENSES|LEGAL COSTS|MAINTENANCE AND PRESERVATION COSTS|TAXES AND INSURANCE|MISCELLANEOUS EXPENSES|ACTUAL LOSS|CUMULATIVE MODIFICATION COSTS|INTEREST RATE STEP INDICATOR|PAYMENT DEFERRAL FLAG|ESTIMATED LOAN-TO-VALUE (ELTV)|ZERO BALANCE REMOVAL UPB|DELINQUENT ACCRUED INTEREST|DELINQUENCY DUE TO DISASTER|BORROWER ASSISTANCE PLAN|CURRENT PERIOD MODIFICATION COSTS|CURRENT INTEREST BEARING UPB|MORTGAGE INSURANCE CANCELLATION INDICATOR|SERVICER NAME|BANKRUPTCY CRAMDOWN COSTS


# Validate the performance schema

In [21]:
performance_columns = performance_header.strip().split("|")

len(performance_columns)



35

In [22]:
performance_file = sample_dir / "performance_sample_file.txt"

performance_file.exists()

True

In [23]:
with open(performance_file, "r") as file:
    first_performance_record = file.readline().strip().split("|")

len(first_performance_record)

35

# Load the performance sample

In [24]:
performance_df = pd.read_csv(
    performance_file,
    sep="|",
    names=performance_columns
)

performance_df.head()

,LOAN IDENTIFIER,PERIOD,CURRENT ACTUAL UPB,CURRENT LOAN DELINQUENCY STATUS,LOAN AGE,REMAINING MONTHS TO LEGAL MATURITY,UNDERWRITING DEFECT AND MAJOR SERVICING DEFECT SETTLEMENT DATE,MODIFICATION FLAG,ZERO BALANCE CODE,ZERO BALANCE EFFECTIVE DATE,...,ESTIMATED LOAN-TO-VALUE (ELTV),ZERO BALANCE REMOVAL UPB,DELINQUENT ACCRUED INTEREST,DELINQUENCY DUE TO DISASTER,BORROWER ASSISTANCE PLAN,CURRENT PERIOD MODIFICATION COSTS,CURRENT INTEREST BEARING UPB,MORTGAGE INSURANCE CANCELLATION INDICATOR,SERVICER NAME,BANKRUPTCY CRAMDOWN COSTS
0,F26Q60000001,200009,52000.0,00,0,360,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,52000.0,7,OTHER,NaN
1,F26Q60000001,200010,52000.0,00,1,359,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,52000.0,7,OTHER,NaN
2,F26Q60000001,200011,52000.0,00,2,358,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,52000.0,7,OTHER,NaN
3,F26Q60000001,200012,52000.0,00,3,357,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,52000.0,7,OTHER,NaN
4,F26Q60000001,200101,52000.0,00,4,356,NaN,NaN,NaN,NaN,...,999,NaN,NaN,NaN,NaN,NaN,52000.0,7,OTHER,NaN


In [25]:
performance_df.shape

(1011, 35)

In [26]:
performance_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1011 entries, 0 to 1010
Data columns (total 35 columns):
 #   Column                                                          Non-Null Count  Dtype  
---  ------                                                          --------------  -----  
 0   LOAN IDENTIFIER                                                 1011 non-null   str    
 1   PERIOD                                                          1011 non-null   int64  
 2   CURRENT ACTUAL UPB                                              1011 non-null   float64
 3   CURRENT LOAN DELINQUENCY STATUS                                 1011 non-null   str    
 4   LOAN AGE                                                        1011 non-null   int64  
 5   REMAINING MONTHS TO LEGAL MATURITY                              1011 non-null   int64  
 6   UNDERWRITING DEFECT AND MAJOR SERVICING DEFECT SETTLEMENT DATE  0 non-null      float64
 7   MODIFICATION FLAG                                             

In [27]:
performance_df.columns.tolist()

['LOAN IDENTIFIER',
 'PERIOD',
 'CURRENT ACTUAL UPB',
 'CURRENT LOAN DELINQUENCY STATUS',
 'LOAN AGE',
 'REMAINING MONTHS TO LEGAL MATURITY',
 'UNDERWRITING DEFECT AND MAJOR SERVICING DEFECT SETTLEMENT DATE',
 'MODIFICATION FLAG',
 'ZERO BALANCE CODE',
 'ZERO BALANCE EFFECTIVE DATE',
 'CURRENT INTEREST RATE',
 'CURRENT NON-INTEREST BEARING UPB',
 'DUE DATE OF LAST PAID INSTALLMENT (DDLPI)',
 'MI RECOVERIES',
 'NET SALES PROCEEDS',
 'NON MI RECOVERIES',
 'TOTAL EXPENSES',
 'LEGAL COSTS',
 'MAINTENANCE AND PRESERVATION COSTS',
 'TAXES AND INSURANCE',
 'MISCELLANEOUS EXPENSES',
 'ACTUAL LOSS',
 'CUMULATIVE MODIFICATION COSTS',
 'INTEREST RATE STEP INDICATOR',
 'PAYMENT DEFERRAL FLAG',
 'ESTIMATED LOAN-TO-VALUE (ELTV)',
 'ZERO BALANCE REMOVAL UPB',
 'DELINQUENT ACCRUED INTEREST',
 'DELINQUENCY DUE TO DISASTER',
 'BORROWER ASSISTANCE PLAN',
 'CURRENT PERIOD MODIFICATION COSTS',
 'CURRENT INTEREST BEARING UPB',
 'MORTGAGE INSURANCE CANCELLATION INDICATOR',
 'SERVICER NAME',
 'BANKRUPTCY CRAM

In [28]:
origination_df["LOAN IDENTIFIER"].nunique()

1000

In [29]:
performance_df["LOAN IDENTIFIER"].nunique()

12

In [30]:
performance_df["LOAN IDENTIFIER"].value_counts().head(10)

LOAN IDENTIFIER
F26Q60000001    250
F26Q60000005    175
F26Q60000006    158
F26Q60000002    118
F26Q60000007     71
F26Q60000008     61
F26Q60000003     52
F26Q60000009     43
F26Q60000012     29
F26Q60000010     26
Name: count, dtype: int64